In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sarima_final = pd.read_csv("../docs/sarima_results_final.csv")
lstm_final = pd.read_csv("../docs/lstm_results_summary.csv")

comparison = sarima_final[['station','order','seasonal_order','MAPE_sarima','RMSE_sarima','MAE_sarima','interpolated_months']].merge(
    lstm_final[['station','MAPE_lstm','RMSE_lstm','MAE_lstm','std_across_runs']], on='station', how='outer'
)

comparison['winner'] = np.where(
    comparison['MAPE_lstm'].isna(), 'SARIMA (LSTM insufficient data)',
    np.where(comparison['MAPE_sarima'] < comparison['MAPE_lstm'], 'SARIMA', 'LSTM')
)

print("=== Overall Statistics ===")
print(f"SARIMA - Mean MAPE: {comparison['MAPE_sarima'].mean():.2f}%  Median: {comparison['MAPE_sarima'].median():.2f}%")
print(f"LSTM   - Mean MAPE: {comparison['MAPE_lstm'].mean():.2f}%  Median: {comparison['MAPE_lstm'].median():.2f}%")
print(f"\nSARIMA - Mean RMSE: {comparison['RMSE_sarima'].mean():.0f}  Mean MAE: {comparison['MAE_sarima'].mean():.0f}")
print(f"LSTM   - Mean RMSE: {comparison['RMSE_lstm'].mean():.0f}  Mean MAE: {comparison['MAE_lstm'].mean():.0f}")

print("\n=== Win Count ===")
print(comparison['winner'].value_counts())

print("\n=== Stations where LSTM actually won ===")
print(comparison[comparison['winner']=='LSTM'][['station','MAPE_sarima','MAPE_lstm']].sort_values('MAPE_lstm'))

comparison.to_csv("../docs/sarima_vs_lstm_comparison.csv", index=False)
print("\nSaved full comparison table.")

=== Overall Statistics ===
SARIMA - Mean MAPE: 8.20%  Median: 7.27%
LSTM   - Mean MAPE: 24.39%  Median: 12.37%

SARIMA - Mean RMSE: 45261  Mean MAE: 37692
LSTM   - Mean RMSE: 81632  Mean MAE: 71748

=== Win Count ===
winner
SARIMA                             46
LSTM                                5
SARIMA (LSTM insufficient data)     3
Name: count, dtype: int64

=== Stations where LSTM actually won ===
                                  station  MAPE_sarima  MAPE_lstm
34                 Etisalat Metro Station     7.873403   6.139688
1     Abu Baker Al Siddique Metro Station     7.274074   6.390319
39              Ibn Battuta Metro Station     9.616211   7.779670
29    Dubai Investment Park Metro Station    13.367488  12.435443
18  Burj Khalifa/Dubai Mall Metro Station          NaN  14.678203

Saved full comparison table.


In [2]:
sarima_final = pd.read_csv("../docs/sarima_results_final.csv")
print(sarima_final[sarima_final['station'].str.contains('Burj Khalifa', na=False)])

                                   station      order seasonal_order  \
17  Burj Khalifa/ Dubai Mall Metro Station  (1, 1, 1)  (1, 1, 1, 12)   

    MAPE_sarima    RMSE_sarima    MAE_sarima  interpolated_months  
17     9.388074  144961.402307  98116.958911                    0  


In [3]:
df_check2 = pd.read_csv("../data_processed/metro_trips_clean.csv")
print("Master file:", [s for s in df_check2['metro_station'].unique() if 'Burj' in s])

df_check3 = pd.read_csv("../data_processed/metro_trips_model_ready.csv")
print("Model-ready file:", [s for s in df_check3['metro_station'].unique() if 'Burj' in s])

Master file: ['Burj Khalifa/Dubai Mall Metro Station']
Model-ready file: ['Burj Khalifa/Dubai Mall Metro Station']


In [4]:
sarima_full_df = pd.read_csv("../docs/sarima_results_final.csv")

# Find the row with the buggy name (regardless of which variant) and fix the name AND the values directly
buggy_mask = sarima_full_df['station'].str.contains('Burj Khalifa', na=False)
print("Rows matching Burj Khalifa (before fix):", sarima_full_df[buggy_mask])

# Overwrite station name to correct spelling
sarima_full_df.loc[buggy_mask, 'station'] = 'Burj Khalifa/Dubai Mall Metro Station'

# Confirm the correct MAPE (9.4%) is already in this row - it is, only the name was wrong
print("\nAfter fixing name only:")
print(sarima_full_df[sarima_full_df['station'] == 'Burj Khalifa/Dubai Mall Metro Station'])

sarima_full_df.to_csv("../docs/sarima_results_final.csv", index=False)
print("\nSaved corrected CSV.")

Rows matching Burj Khalifa (before fix):                                    station      order seasonal_order  \
17  Burj Khalifa/ Dubai Mall Metro Station  (1, 1, 1)  (1, 1, 1, 12)   

    MAPE_sarima    RMSE_sarima    MAE_sarima  interpolated_months  
17     9.388074  144961.402307  98116.958911                    0  

After fixing name only:
                                  station      order seasonal_order  \
17  Burj Khalifa/Dubai Mall Metro Station  (1, 1, 1)  (1, 1, 1, 12)   

    MAPE_sarima    RMSE_sarima    MAE_sarima  interpolated_months  
17     9.388074  144961.402307  98116.958911                    0  

Saved corrected CSV.


In [5]:
sarima_final = pd.read_csv("../docs/sarima_results_final.csv")
lstm_final = pd.read_csv("../docs/lstm_results_summary.csv")

comparison = sarima_final[['station','order','seasonal_order','MAPE_sarima','RMSE_sarima','MAE_sarima','interpolated_months']].merge(
    lstm_final[['station','MAPE_lstm','RMSE_lstm','MAE_lstm','std_across_runs']], on='station', how='outer'
)

comparison['winner'] = np.where(
    comparison['MAPE_lstm'].isna(), 'SARIMA (LSTM insufficient data)',
    np.where(comparison['MAPE_sarima'] < comparison['MAPE_lstm'], 'SARIMA', 'LSTM')
)

print("=== Overall Statistics ===")
print(f"SARIMA - Mean MAPE: {comparison['MAPE_sarima'].mean():.2f}%  Median: {comparison['MAPE_sarima'].median():.2f}%")
print(f"LSTM   - Mean MAPE: {comparison['MAPE_lstm'].mean():.2f}%  Median: {comparison['MAPE_lstm'].median():.2f}%")

print("\n=== Win Count ===")
print(comparison['winner'].value_counts())

print("\n=== Stations where LSTM actually won ===")
print(comparison[comparison['winner']=='LSTM'][['station','MAPE_sarima','MAPE_lstm']].sort_values('MAPE_lstm'))

# Sanity check - confirm Burj Khalifa now shows correctly
print("\n=== Burj Khalifa check ===")
print(comparison[comparison['station'].str.contains('Burj Khalifa', na=False)])

comparison.to_csv("../docs/sarima_vs_lstm_comparison.csv", index=False)
print("\nSaved corrected full comparison table.")

=== Overall Statistics ===
SARIMA - Mean MAPE: 8.20%  Median: 7.27%
LSTM   - Mean MAPE: 24.39%  Median: 12.37%

=== Win Count ===
winner
SARIMA                             47
LSTM                                4
SARIMA (LSTM insufficient data)     2
Name: count, dtype: int64

=== Stations where LSTM actually won ===
                                station  MAPE_sarima  MAPE_lstm
33               Etisalat Metro Station     7.873403   6.139688
1   Abu Baker Al Siddique Metro Station     7.274074   6.390319
38            Ibn Battuta Metro Station     9.616211   7.779670
28  Dubai Investment Park Metro Station    13.367488  12.435443

=== Burj Khalifa check ===
                                  station      order seasonal_order  \
17  Burj Khalifa/Dubai Mall Metro Station  (1, 1, 1)  (1, 1, 1, 12)   

    MAPE_sarima    RMSE_sarima    MAE_sarima  interpolated_months  MAPE_lstm  \
17     9.388074  144961.402307  98116.958911                    0  14.678203   

        RMSE_lstm     MAE_lst